# 3D Heat Equation for a Gaussian Heat Source

This notebook demonstrates the generation and analysis of a three-dimensional heat source, and solution to the heat equation using the Alternate Direction Implicit Crank-Nicolson discretization method of Douglas and Rachford.


## 1. Import libraries and generate heat sources
- Refer to NoteBook 1 for a refresher on importing modules.
- In this notebook we will see how the 3D methods inherit from their 2D ancestors.


<div class="alert alert-block alert-info"> **Info:** Here we create an instance of the 3D `photonicslab.heateqn.HeatEqn3D` object and use the `get_explicit_fd_params` method [overridden](https://docs.python.org/2/tutorial/classes.html "Python Classes Turorial") from the parent class `HeatEqn2D`.</div>

In [ ]:
import os, sys
module_path = os.path.abspath(os.path.join('../'))
if module_path not in sys.path:
    sys.path.append(module_path)

import importlib
import photonicslab

importlib.reload(photonicslab.heateqn)
#mygauss3dQ = photonicslab.heateqn.HeatEqn3D(nx=110, ny=100, nz=150, nt=50, Lx=5e-3, Ly=5e-3, Lz=1e-1, \
#                                                    hdf5grp='nb8', cavity_mode_waist=191.5e-6)
mygauss3dQ = photonicslab.heateqn.HeatEqn3D(nx=100, ny=100, nz=50, nt=50, Lx=0.5, Ly=0.5, Lz=1., \
                                            Qmax=800, hdf5grp='nb8', cavity_mode_waist=191.5e-3)
#mygauss3dQ.add_normalized_pump_gaussian(alpha=12., center_x=-5e-4)
#mygauss3dQ.add_normalized_pump_gaussian(alpha=12., center_x=5e-4)
mygauss3dQ.add_normalized_pump_gaussian(alpha=0.12)
mygauss3dQ.plot_source_profile(savepath='images/nb8_sourceq.png', z_ind=-1)
mygauss3dQ.get_cn_fd_params()

<img width="800" height="600" src="images/nb8_sourceq.png">

<div class="alert alert-block alert-success"> **Tip:** Many of the modules are built with exception handling. Should you try to visualize the cross-sectional heat profile of the crystal / lens beyond the extent of it's physical axial length, a `ValueError` exception will be raised. And remember to press `F5` to refresh the page if the image is not updating.</div>

## 2. Solve and Visualize 3D heat equation using various finite difference methods

- Create `FDSolver3D` object using the integration parameters configured with the `mygauss3dQ` object.
- Specify unique paths for thermal and source profiles in database.
- For comparison and benchmarking, configurations for the explicit and implicit methods are also prepared.

<div class="alert alert-block alert-danger"> **Warning:** If you've generated other profiles in corresponding notebooks for later analysis, ensure that you use different paths fo the thermal and phase profiles in the in the hierarchical data format `storage.hdf5` file.</div>

### 2.1. Alternating Direction Implicit Method

This is the optimal solution, and in the subsequent notebooks GPU-accelerated solutions of this implimentation will be explored.

In [ ]:
importlib.reload(photonicslab.fdsolver)
myfdsolver3d = photonicslab.fdsolver.FDSolver3D(hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb8')
myfdsolver3d.adi_method()

importlib.reload(photonicslab.visualize)
myview3d = photonicslab.visualize.Visualize3D(hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb8')
myview3d.steadystate_profiles(savepath='images/nb8_steady_state.png')
myview3d.transient_profiles(savepath='videos/nb8_transient.mp4')

<img width="800" height="600" src="images/nb8_steady_state.png">
<video width="640" height="480" controls autoplay loop src="videos/nb8_transient.mp4"/>

## 3D phase modulated (VarOptic) Gaussian beam generation and propagation

* We generated a transmission function corresponding to thermal-profile related phase modulation.


* We create a seed beam propagation object called `mybeam` and give the initial parameters:

    - Initial mesh grid spacing \ layout.
    - The time evolution matrix.
    - The transient solution to the heat equation.
    - The normalised (to fd coefficients) initial heat load.

* Generate and Plot seed beam profile.

* Generate and Play animation (loop) for temporal seed beam evolution at fixed z.

<div class="alert alert-block alert-info"> **Info:** Due to the numerics of the convolution function, it will show signs of breakdown near the boundaries. To circumvent this use a bigger meshgrid.</div>

In [2]:
importlib.reload(photonicslab.phasemod)
myphasemod3d = photonicslab.phasemod.ThermalLens3D(hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb8')

myphasemod3d.phase(hdf5table='temperature3d', hdf5dset_phase='phase3d')
myphasemod3d.phase_heat(hdf5table='temperature3d', hdf5dset_phase='phase_heat')

In [3]:
importlib.reload(photonicslab.visualize)
myphaseview3d = photonicslab.visualize.VisualizePhase3D(hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb8')
myphaseview3d.steadystate_profiles(savepath='images/nb8_steady_state_phase.png')
myphaseview3d.transient_profiles(savepath='videos/nb8_transient_phase.mp4')

myphase_heatview3d = photonicslab.visualize.VisualizePhase3D(hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb8', \
                                                            hdf5table='phase_heat')
myphase_heatview3d.steadystate_profiles(savepath='images/nb8_steady_state_phase_heat.png')
myphase_heatview3d.transient_profiles(savepath='videos/nb8_transient_phase_heat.mp4')

/usr/lib/python3.7/site-packages/matplotlib/colorbar.py:601: UserWarning: set_ticks() must have been called.
  warnings.warn("set_ticks() must have been called.")


transient_profiles took 503.31054186820984 seconds.


/usr/lib/python3.7/site-packages/matplotlib/colorbar.py:601: UserWarning: set_ticks() must have been called.
  warnings.warn("set_ticks() must have been called.")


transient_profiles took 75.08988380432129 seconds.


<img width="800" height="600" src="images/nb8_steady_state_phase.png"/>
<video width="640" height="480" controls autoplay loop src="videos/nb8_transient_phase.mp4"/>

<img width="800" height="600" src="images/nb8_steady_state_phase_heat.png"/>
<video width="640" height="480" controls autoplay loop src="videos/nb8_transient_phase_heat.mp4"/>

In [4]:
importlib.reload(photonicslab.beamprop)

mybeam3d = photonicslab.beamprop.BeamProp3D(wavelen=1.064e-4, waist=191.5e-4, p=0, l=0, z=0, phi=0, \
                                            hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb8', \
                                            hdf5table_phase='phase3d', hdf5subgrp_beam='beam3d', \
                                            hdf5table_intensity='tem00_intensity3d')

/usr/lib/python3.7/site-packages/scipy/fftpack/basic.py:160: FutureWarning: Using a non-tuple sequence for multidimensional indexing is deprecated; use `arr[tuple(seq)]` instead of `arr[seq]`. In the future this will be interpreted as an array index, `arr[np.array(seq)]`, which will result either in an error or a different result.
  z[index] = x


In [5]:
importlib.reload(photonicslab.visualize)
mybeamview3d = photonicslab.visualize.VisualizeBeam3D(hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb8', \
                                                      hdf5subgrp_beam='beam3d', hdf5table='tem00_intensity3d')
mybeamview3d.steadystate_profiles(savepath='images/nb8_steady_state_tem00_intensity.png')
#mybeamview3d.steadystate_profiles(z_dist=30, savepath='images/nb8_steady_state_beam_z30.png')
#mybeamview3d.steadystate_profiles(z_dist=50, savepath='images/nb8_steady_state_beam_z50.png')
#mybeamview3d.steadystate_profiles(z_dist=100, savepath='images/nb8_steady_state_beam_z100.png')

mybeamview3d.transient_profiles(savepath='videos/nb8_transient_tem00_intensity.mp4')

transient_profiles took 145.1461102962494 seconds.


<img width="800" height="600" src="images/nb8_steady_state_tem00_intensity.png"/>
<video width="640" height="480" controls autoplay loop src="videos/nb8_transient_tem00_intensity.mp4"/>

In [8]:
importlib.reload(photonicslab.beamprop)

mybeam3d = photonicslab.beamprop.BeamProp3D(wavelen=1.064e-4, waist=191.5e-4, p=0, l=0, z=0, phi=0, \
                                            hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb8', \
                                            hdf5table_phase='phase_heat', hdf5subgrp_beam='beam_heat', \
                                            hdf5table_intensity='tem00_intensity_heat')

/usr/lib/python3.7/site-packages/scipy/fftpack/basic.py:160: FutureWarning: Using a non-tuple sequence for multidimensional indexing is deprecated; use `arr[tuple(seq)]` instead of `arr[seq]`. In the future this will be interpreted as an array index, `arr[np.array(seq)]`, which will result either in an error or a different result.
  z[index] = x


In [9]:
importlib.reload(photonicslab.visualize)
mybeamview3d = photonicslab.visualize.VisualizeBeam3D(hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb8', \
                                                     hdf5subgrp_beam='beam_heat', hdf5table='tem00_intensity_heat')
mybeamview3d.steadystate_profiles(savepath='images/nb8_steady_state_tem00_intensity_heat.png')
mybeamview3d.transient_profiles(savepath='videos/nb8_transient_tem00_intensity_heat.mp4')

transient_profiles took 92.54581069946289 seconds.


<img width="800" height="600" src="images/nb8_steady_state_tem00_intensity_heat.png"/>
<video width="640" height="480" controls autoplay loop src="videos/nb8_transient_tem00_intensity_heat.mp4"/>